# 16 · โครงงาน: จำแนกภาพตัวเลขและนำโมเดลกลับมาใช้

[กลับสารบัญ](../../README.md) · [Notebook](lesson.ipynb) · [Python](lesson.py)

เวลาเรียนและทดลองโดยประมาณ: 90–150 นาที  
สิ่งที่จะทำได้: ทำ workflow ตั้งแต่โจทย์จนถึง model artifact พร้อมรายงานผล ข้อจำกัด และ input contract ได้

## ขอบเขตโครงงาน

ทำนายตัวเลข 0–9 จากภาพ grayscale 8×8 ในชุด digits ที่มากับ scikit-learn แต่ละภาพเป็น 64 pixel ค่า 0–16 ไม่ใช่ MNIST 28×28 และไม่ใช่ระบบอ่านเอกสารหรือภาพถ่ายทั่วไป

เกณฑ์หลักคือ macro F1 เพื่อให้แต่ละคลาสมีน้ำหนักเท่ากัน รายงาน accuracy ควบคู่ และใช้ DummyClassifier เป็น baseline กำหนด grid ก่อนดู test แบ่ง test 20% แบบ stratified แล้วเลือก C/gamma ของ SVC ด้วย 3-fold CV บน train

## สิ่งที่ส่งออก

โปรแกรมบันทึก Pipeline ทั้ง scaler และ SVC, metrics.json, index ของ train/test, input CSV ตัวอย่าง, confusion matrix และภาพข้อผิดพลาดถ้ามี จากนั้นโหลดโมเดลที่เพิ่งสร้างแล้วตรวจว่าทายเหมือนเดิม การเก็บ split index ช่วยตรวจว่าใช้ตัวอย่างใดประเมิน

Input contract กำหนด array สองมิติ มี 64 คอลัมน์อย่างน้อยหนึ่งแถว เป็น finite และอยู่ช่วง 0–16 ภาพใหม่ต้องผ่านขั้นตอนเตรียมภาพที่ตรงกับข้อมูลฝึกด้วย ไม่ใช่แค่ resize แล้วสรุปว่าใช้ได้

ไฟล์ joblib ใช้กลไก pickle ซึ่งเรียกโค้ดได้ระหว่างโหลด จึงโหลดเฉพาะ artifact ที่สร้างเองหรือเชื่อถือ และใช้รุ่นไลบรารีตาม environment เดิม อ่าน [model card](../../docs/model-card.md) เพื่อบันทึกขอบเขตการใช้งาน

## ทำนายหลังฝึกเสร็จ

จากโฟลเดอร์หลัก รันหลัง lesson.py:

```bash
uv run --frozen python lessons/16-digits-project/predict.py outputs/16-digits-project/example-input.csv
```

CSV ต้องไม่มี header และเรียง pixel แบบ row-major ตามภาพ 8×8 ตัวอย่างนี้มาจาก holdout เพื่อแสดงการโหลดโมเดล ไม่ใช่การประเมินกับข้อมูลภายนอก

## ลงมือรัน

จากโฟลเดอร์หลักของ repo:

```bash
uv run --frozen python lessons/16-digits-project/lesson.py
```

หากใช้ venv ให้ใช้ `python` แทน `uv run --frozen python` บล็อกนี้รันได้ทั้งชุดโดยไม่ต้องรันบทก่อนหน้า



In [ ]:
# ใช้โฟลเดอร์ repo เป็น working directory เมื่อเปิด notebook ในโฟลเดอร์บทเรียน
from pathlib import Path
import os

here = Path.cwd()
root = next((p for p in [here, *here.parents]
             if (p / "pyproject.toml").is_file() and (p / "lessons").is_dir()), here)
os.chdir(root)
print("Working directory:", root)


In [ ]:
"""โครงงานจำแนกภาพตัวเลข: เลือกโมเดล รายงานข้อผิดพลาด บันทึก Pipeline"""


### การตรวจ schema ใช้ร่วมกันทั้งตอนฝึกและทำนาย


In [ ]:
import json
import platform
from pathlib import Path

import joblib
import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def validate_pixels(values):
    array = np.asarray(values, dtype=np.float64)
    if array.ndim != 2 or array.shape[1] != 64 or len(array) == 0:
        raise ValueError("Expected shape (n_samples, 64) with at least one sample")
    if not np.isfinite(array).all() or np.any((array < 0) | (array > 16)):
        raise ValueError("Pixels must be finite and in the interval [0, 16]")
    return array


def build_search():
    return GridSearchCV(
        Pipeline([("scale", StandardScaler()), ("svc", SVC())]),
        {"svc__C": [1.0, 10.0], "svc__gamma": ["scale", 0.01]},
        scoring="f1_macro", cv=StratifiedKFold(3, shuffle=True, random_state=42),
        n_jobs=1,
    )


def main():
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay

    data = load_digits()
    X = validate_pixels(data.data)
    train_idx, test_idx = train_test_split(
        np.arange(len(X)), test_size=0.2, stratify=data.target, random_state=42
    )
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = data.target[train_idx], data.target[test_idx]
    baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    search = build_search().fit(X_train, y_train)
    pred = search.predict(X_test)
    report = {
        "dataset": "sklearn load_digits; 8x8 pixels; range 0..16",
        "split": "stratified random holdout 20%; seed 42; not a writer-group split",
        "train_rows": len(train_idx), "test_rows": len(test_idx),
        "baseline_accuracy": accuracy_score(y_test, baseline.predict(X_test)),
        "best_params": search.best_params_, "cv_macro_f1": search.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_macro_f1": f1_score(y_test, pred, average="macro"),
        "classification_report": classification_report(y_test, pred, output_dict=True),
        "python": platform.python_version(), "sklearn": sklearn.__version__,
    }
    print(json.dumps(report, indent=2))
    out = Path("outputs/16-digits-project")
    out.mkdir(parents=True, exist_ok=True)
    joblib.dump(search.best_estimator_, out / "model.joblib")
    (out / "metrics.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    np.savez(out / "split-indices.npz", train=train_idx, test=test_idx)
    np.savetxt(out / "example-input.csv", X_test[:3], delimiter=",")
    restored = joblib.load(out / "model.joblib")  # โหลดเฉพาะไฟล์ที่เพิ่งสร้างเอง
    np.testing.assert_array_equal(pred, restored.predict(X_test))
    errors = np.flatnonzero(pred != y_test)
    print("Misclassified held-out examples:", len(errors))
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title("Final holdout confusion matrix")
    fig.tight_layout()
    fig.savefig(out / "confusion-matrix.png", dpi=140)
    plt.show()
    plt.close(fig)
    if len(errors):
        fig, axes = plt.subplots(1, min(6, len(errors)), figsize=(10, 2), squeeze=False)
        for ax, i in zip(axes[0], errors[:6]):
            ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
            ax.set_title(f"true={y_test[i]}, pred={pred[i]}")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(out / "errors.png", dpi=140)
        plt.show()
        plt.close(fig)




### รันโครงงานตั้งแต่ต้น


In [ ]:
if __name__ == "__main__":
    main()




## อ่านผลที่ได้

เกิด outputs/16-digits-project/model.joblib, metrics.json, split-indices.npz, example-input.csv และกราฟ ผลจากโมเดลที่โหลดกลับต้องตรงกับโมเดลก่อนบันทึก ค่าทศนิยมอาจต่างเล็กน้อยตามแพลตฟอร์ม

## ภาพตัวอย่างจากการรัน

![ผลการทดลอง 16-digits-project](../../assets/figures/16-digits-project-confusion-matrix.png)

ภาพนี้สร้างจากโค้ดและ seed ในบท เพื่อช่วยอ่านผลก่อนทดลองด้วยตัวเอง

## แบบฝึกหัด

ทำรายงานโครงงานหนึ่งหน้าให้มีโจทย์ baseline วิธี split metric หลัก ผลข้อผิดพลาด และข้อจำกัดอย่างน้อย 3 ข้อ โดยใช้ docs/model-card.md เป็นแนวทาง

<details>
<summary>แนวคำตอบ — ลองทำก่อนเปิดอ่าน</summary>

ข้อจำกัดสำคัญคือภาพขนาด/ช่วงค่าคงที่ การ split ไม่ได้แยกตามผู้เขียน และยังไม่มีการทดสอบภาพภายนอก ถ้านำ error analysis ไปปรับโมเดลต่อ ให้จัด validation ใหม่และกัน test ใหม่สำหรับข้อสรุปสุดท้าย ไม่รายงานผลจากการเลือกซ้ำบน test เดิมเป็น unbiased estimate

</details>

## อ่านต่อ

- [scikit-learn: load_digits](https://scikit-learn.org/1.7/modules/generated/sklearn.datasets.load_digits.html)
- [Model persistence](https://scikit-learn.org/1.7/model_persistence.html)
- [Handbook: Application pipeline](https://jakevdp.github.io/PythonDataScienceHandbook/05.14-image-features.html)

